In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def _get_font():
    # Register Arial first when an Arial font file is available
    for p in [
        "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf",
        "/usr/share/fonts/truetype/msttcorefonts/arial.ttf",
        "/Library/Fonts/Arial.ttf",
        "C:/Windows/Fonts/arial.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)

    for name in ["Arial", "Helvetica"]:
        try:
            path = fm.findfont(name, fallback_to_default=False)
            real_name = fm.FontProperties(fname=path).get_name()
            if real_name in ["Arial", "Helvetica"]:
                return real_name
        except Exception:
            pass

    raise RuntimeError("Arial/Helvetica not found. Install one of them before final export.")

def setup_style(double_column: bool = True):
    font_family = _get_font()

    plt.rcParams.update({
        "figure.dpi": 600,
        "savefig.dpi": 600,
        "savefig.bbox": None,
        "savefig.pad_inches": 0.02,

        "font.family": "sans-serif",
        "font.sans-serif": [font_family],
        "text.usetex": False,

        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",

        "axes.labelsize": 6,
        "xtick.labelsize": 5,
        "ytick.labelsize": 5,
        "legend.fontsize": 5,
        "legend.title_fontsize": 5,

        "axes.linewidth": 0.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.width": 0.4,
        "ytick.major.width": 0.4,
        "xtick.major.size": 2.2,
        "ytick.major.size": 2.2,

        "axes.grid": False,
        "patch.edgecolor": "none",
        "patch.linewidth": 0.0,
        "axes.titlepad": 4,
        "axes.labelpad": 3,
    })


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.contingency_tables import mcnemar

# ============ 0. Define statistical test functions (new) ============
def _calculate_p_raw(df, metric_col, test_type='wilcoxon', condition_col='condition', id_col='hadm_id'):
    """只返回 raw p-value（方便后面集中做 Holm）"""
    pivot_df = df.pivot(index=id_col, columns=condition_col, values=metric_col).dropna()
    if len(pivot_df) < 2:
        return np.nan
    group1 = pivot_df["Baseline (grounded)"]
    group2 = pivot_df["Perturbed (ungrounded)"]
    if test_type == 'wilcoxon':
        # For continuous variables (confidence scores)
        stat, p_val = wilcoxon(group1, group2)
        return float(p_val)
    
    elif test_type == 'mcnemar':
        # For binary variables (accuracy: 0/1)
        # Construct the contingency table.
        #        Perturbed 0   Perturbed 1
        # Base 0    a             b
        # Base 1    c             d
        # crosstab does not fill missing categories by default, so manual construction is safer
        y_true = group1.astype(int)
        y_pred = group2.astype(int)
        # table[0,0](0->0), [0,1](0->1), [1,0](1->0), [1,1](1->1)
        # McNemar only uses b and c (the discordant pairs)
        # Using sklearn or statsmodels here is more convenient than calculating manually.
        # For simplicity, use statsmodels mcnemar here; it requires a crosstab
        ct = pd.crosstab(y_true, y_pred)
        # Ensure a 2x2 table; crosstab may omit a cell when its count is zero
        if ct.shape != (2,2):
             # Rare case: for example, when there is no change, p = 1.0
             return 1.0 
        res = mcnemar(ct, exact=True) if ct.values.min() < 25 else mcnemar(ct, exact=False)
        return res.pvalue

    return np.nan

def format_p_text(p, stars=True):
    if np.isnan(p):
        return "N/A"
    if p < 0.001:
        return "P < 0.001 ***" if stars else "P < 0.001"
    elif p < 0.01:
        return "P < 0.01 **" if stars else "P < 0.01"
    elif p < 0.05:
        return f"P = {p:.3f} *" if stars else f"P = {p:.3f}"
    else:
        return f"P = {p:.3f} (ns)"



In [33]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# ============ 1. Read and combine data ============
# Note: the files appear tab-separated (.tsv style), so use sep='\t' here
control_path = "/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/evaluation_logs/63/5runs/63.10/master_feature_dataframe.csv"
halluc_path = "/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/evaluation_logs/64/64.10/master_feature_dataframe.csv"

df_control = pd.read_csv(control_path)
df_hallucination = pd.read_csv(halluc_path)

# Add a condition column for each dataset.
df_control["condition"] = "Baseline (grounded)"
df_hallucination["condition"] = "Perturbed (ungrounded)"

# Merge the two datasets.
df_combined = pd.concat([df_control, df_hallucination], ignore_index=True)

# ============ 2. Preprocessing: disease labels / long-format structure ============

# If the dataset column already contains clean disease labels (such as appendicitis and pneumonia),
# use it directly as disease, or apply a mapping to improve display names
def clean_disease(x: str) -> str:
    x = str(x).strip()
    # Apply simple formatting here (capitalize the first letter); a custom mapping can also be used
    return x.replace("_", " ").title()

df_combined["disease"] = df_combined["dataset"].apply(clean_disease)

# Confidence metrics to plot.
metrics_to_plot = [
    "Consistency_Dx",
    "ProbScore_Dx",
    "LingCert_R",
]

# Wide table -> long table (tidy data) for easier seaborn faceting
df_long = df_combined.melt(
    id_vars=["disease", "condition"],
    value_vars=metrics_to_plot,
    var_name="metric",
    value_name="score",
)

print("处理后的长表数据预览：")
print(df_long.head())


处理后的长表数据预览：
        disease            condition          metric     score
0  Appendicitis  Baseline (grounded)  Consistency_Dx  0.973357
1  Appendicitis  Baseline (grounded)  Consistency_Dx  0.918101
2  Appendicitis  Baseline (grounded)  Consistency_Dx  1.000000
3  Appendicitis  Baseline (grounded)  Consistency_Dx  0.973357
4  Appendicitis  Baseline (grounded)  Consistency_Dx  1.000000


In [ ]:
# ============ 3. Visualization: faceted slope plot (confidence metrics) ============

# white background, thin axes, and no prominent grid
sns.set_style("white")
setup_style()

condition_order = ["Baseline (grounded)", "Perturbed (ungrounded)"]
metric_order = metrics_to_plot
# Ensure a fixed condition order (Control on the left, Hallucination on the right)
df_long["condition"] = pd.Categorical(
    df_long["condition"],
    categories=condition_order,
    ordered=True
)

g = sns.relplot(
    data=df_long,
    x="condition",
    y="score",
    hue="disease",
    col="metric",
    col_order=metric_order,
    kind="line",
    col_wrap=3,
    marker="o",
    height=3.2,          # Slightly shorter, like a small paper panel.
    aspect=1.1,
    legend="full",
    facet_kws={"sharey": False},
    sort=False,
    estimator="mean",
    errorbar=None,
)
g.fig.set_size_inches(180 / 25.4, 65 / 25.4)


# ============ 4. Styling and detail adjustments with P-values ============

# 1) Remove the small title above each subplot.
g.set_titles("")

# Clear x-axis labels for all subplots here.
g.set_xlabels("")

# 2) Move each metric name to its subplot y-axis.
metric_pretty_names = {
    "Consistency_Dx": "Consistency_Dx",
    "ProbScore_Dx": "ProbScore_Dx",
    "LingCert_R": "LingCert_R",
}

##############################################################
from statsmodels.stats.multitest import multipletests

# Apply Holm only to the three Fig. 5b metrics (one family)
raw_ps = []
for m in metrics_to_plot:
    raw_ps.append(_calculate_p_raw(df_combined, metric_col=m, test_type='wilcoxon'))

# Holm correction (automatically handles sorting)
# Note: filter NaN values first; otherwise multipletests will fail
# valid = [(m, p) for m, p in zip(metrics_family, raw_ps) if not np.isnan(p)]
# valid_metrics = [x[0] for x in valid]
# valid_ps = [x[1] for x in valid]

# p_adjust_map = {}
# if len(valid_ps) > 0:
#     _, p_holm, _, _ = multipletests(valid_ps, method="holm")
#     p_adjust_map = {m: float(ph) for m, ph in zip(valid_metrics, p_holm)}

# Perform Holm correction.
valid_indices = [i for i, p in enumerate(raw_ps) if not np.isnan(p)]
valid_ps = [raw_ps[i] for i in valid_indices]

p_adjust_map = {}
if valid_ps:
    _, p_holm, _, _ = multipletests(valid_ps, method="holm")
    for idx, p_adj in zip(valid_indices, p_holm):
        p_adjust_map[metrics_to_plot[idx]] = p_adj

##############################################################


for ax, metric in zip(g.axes.flat, metric_order):
    # Write the corresponding metric in the left y-axis label.
    ax.set_ylabel(metric_pretty_names.get(metric, metric), fontsize=6)
    
    # ax.set_ylabel(metric, fontsize=12)
    # Use a shared x-axis label but show it only on the bottom row; remove this condition to show it on all plots.
    # ax.set_xlabel("Experimental condition", fontsize=11)

    # 2. Calculate and add P-values.
    # Get the adjusted P-value
    p_adj = p_adjust_map.get(metric, np.nan)
    # Show only P_adj for a concise presentation.
    p_text = f"P_adj {format_p_text(p_adj)}"

    
    # Add the P-value inside the upper-left corner of the plot
    # transform=ax.transAxes means (0,0) is bottom-left and (1,1) is top-right
    ax.text(
        0.05, 0.95, p_text, 
        transform=ax.transAxes, 
        fontsize=5, 
        fontweight='bold', 
        va='top', ha='left',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.8) # Add a white background to prevent overlap.
    )

    # remove top and right spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Lighten the grid, keeping only subtle y-axis gridlines or disabling it entirely
    ax.grid(True, axis="y", linewidth=0.5, alpha=0.2)
    ax.grid(False, axis="x")

    # Use lighter tick styling.
    ax.tick_params(axis="both", which="both", width=0.5, labelsize=5)
    # ax.set_xlim(-0.03, 1)
    # ax.set_xticks([0, 1])
    ax.set_xticklabels(
        ["Baseline\n(grounded)", "Perturbed\n(ungrounded)"],
        fontsize=5,
    )


# 3) Legend: bold title, positioned outside on the right.
g.legend.set_title("Disease", prop={
    # "weight": "bold", 
    "size": 5})
for t in g.legend.get_texts():
    t.set_fontsize(5)

# Move the legend to the upper-right side to avoid covering subplots
sns.move_legend(
    g,
    "center left",
    bbox_to_anchor=(0.85, 0.5),
    borderaxespad=0,
    frameon=False,
)
for handle in g.legend.legend_handles:
    handle.set_markersize(2)
    handle.set_linewidth(0.6)


# 4) Avoid a large suptitle; adjust only the layout
plt.tight_layout(rect=[0, 0, 0.85, 1])

# Export Plot 1
import os

output_dir="/mnt/bulk-sirius/lizhang/LiWS/Medical_Llama_Agents/data/plots/64/64.10/hallucination"
os.makedirs(output_dir, exist_ok=True)

base_filename: str = "Faceted_Slopegraph"
base = os.path.join(output_dir, base_filename)
# plt.savefig(base + ".svg", format="svg", dpi=600, bbox_inches='tight')
# plt.savefig(base + ".png", format="png", dpi=600, bbox_inches='tight')
plt.savefig(base + ".pdf", format="pdf", dpi=600, bbox_inches=None)
plt.savefig(base + ".svg", format="svg", dpi=600, bbox_inches=None)
plt.savefig(base + ".png", format="png", dpi=600, bbox_inches=None)

plt.close()
print(f"✅ Saved Confidence Plot with P-values")


/tmp/ipykernel_2597512/2739568528.py:122: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
/tmp/ipykernel_2597512/2739568528.py:122: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
/tmp/ipykernel_2597512/2739568528.py:122: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(


✅ Saved Confidence Plot with P-values


In [40]:
# ============ 5. Visualization: accuracy slopegraph (with P-values) ============

# Convert booleans to int (0/1) for the Wilcoxon test
# For 0/1 data, the Wilcoxon test detects distribution shifts and is suitable for paired binary data
if df_combined["final_decision"].dtype == object:
    df_combined["final_decision_numeric"] = df_combined["final_decision"].astype(str).str.lower().map({"true": 1, "false": 0})
else:
    df_combined["final_decision_numeric"] = df_combined["final_decision"].astype(int)

# Calculate the McNemar P-value.
p_acc = _calculate_p_raw(df_combined, metric_col="final_decision_numeric", test_type='mcnemar')

# Calculate aggregated data for plotting
df_acc = (
    df_combined
    .groupby(["disease", "condition"], observed=False)
    .agg(accuracy=("final_decision_numeric", "mean"))
    .reset_index()
)
df_acc["accuracy"] = df_acc["accuracy"] * 100 # Convert to percentages.

print("Accuracy table preview:")
print(df_acc["condition"].value_counts())
print(df_acc[["disease", "condition", "accuracy"]].head(20))



Accuracy table preview:
condition
Baseline (grounded)       7
Perturbed (ungrounded)    7
Name: count, dtype: int64
               disease               condition   accuracy
0         Appendicitis     Baseline (grounded)  97.972973
1         Appendicitis  Perturbed (ungrounded)  93.918919
2        Cholecystitis     Baseline (grounded)  90.697674
3        Cholecystitis  Perturbed (ungrounded)  80.620155
4       Diverticulitis     Baseline (grounded)  94.444444
5       Diverticulitis  Perturbed (ungrounded)  75.925926
6         Pancreatitis     Baseline (grounded)  92.307692
7         Pancreatitis  Perturbed (ungrounded)  59.615385
8            Pneumonia     Baseline (grounded)  51.724138
9            Pneumonia  Perturbed (ungrounded)  44.827586
10  Pulmonary Embolism     Baseline (grounded)  88.888889
11  Pulmonary Embolism  Perturbed (ungrounded)  53.333333
12                 Uti     Baseline (grounded)  87.755102
13                 Uti  Perturbed (ungrounded)  22.448980


In [ ]:
# Plot
setup_style()

fig, ax = plt.subplots(figsize=(60 / 25.4, 65 / 25.4), dpi=600)

sns.lineplot(
    data=df_acc,
    x="condition",
    y="accuracy",
    hue="disease",
    marker="o",
    sort=False,
    ax=ax,
    legend=False # Since the legend is already above, this can usually be omitted or placed separately.
)

# Calculate the accuracy P-value.
# Annotate the McNemar P-value
# This is a single hypothesis, so no Holm correction is needed; show P directly
ax.text(
    0.05, 0.95, f"Global P {format_p_text(p_acc)}", 
    transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left'
)

# details
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(True, axis="y", linewidth=0.5, alpha=0.2)
ax.set_xlabel("")
ax.set_ylabel("Diagnostic Accuracy (%)", fontsize=6)
ax.set_ylim(0, 105)

plt.tight_layout()

base = os.path.join(output_dir, "accuracy_slopegraph")
# plt.savefig(base + ".svg", format="svg", dpi=600, bbox_inches='tight')
plt.savefig(base + ".pdf", format="pdf", dpi=600, bbox_inches=None)
plt.savefig(base + ".svg", format="svg", dpi=600, bbox_inches=None)

plt.close()
print(f"✅ Saved Accuracy Plot with P-value")

✅ Saved Accuracy Plot with P-value
